**1. Write a PyTorch program to implement a basic feed-forward neural network (multi-layer perceptron) for a handwritten
digit classification task. The program should load a handwritten digit dataset, split it into training and testing sets (if
not already provided), normalize the images, and convert the data into tensors. Each image must be flattened into a
1-D vector before being passed to the network.**

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Hyperparameters
input_size = 28 * 28      # MNIST images are 28x28
hidden_size = 128
num_classes = 10
batch_size = 64
learning_rate = 0.001
num_epochs = 5

# 2. Data Loading & Preprocessing

# Transform: Convert to tensor + Normalize
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts image to tensor (C x H x W) and scales to [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # Mean and std for MNIST
])

# Download MNIST dataset
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    transform=transform,
    download=True
)

# DataLoader (handles batching and shuffling)
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=batch_size,
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=batch_size,
                         shuffle=False)

# 3. Define the Neural Network
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.flatten(x)   # Flatten 28x28 -> 784
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Instantiate model
model = MLP(input_size, hidden_size, num_classes)

# 4. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 5. Training Loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

# 6. Testing the Model
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nTest Accuracy: {100 * correct / total:.2f}%")

Epoch [1/5], Loss: 0.2592
Epoch [2/5], Loss: 0.1150
Epoch [3/5], Loss: 0.0799
Epoch [4/5], Loss: 0.0621
Epoch [5/5], Loss: 0.0485

Test Accuracy: 97.51%


**2. Design a neural network consisting of an input layer, one hidden layer with a ReLU activation function, and an output
layer corresponding to the 10 digit classes (0–9). You must define the model by inheriting from torch.nn.Module and
manually implement the forward() function.**

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DigitClassifier(nn.Module):
    def __init__(self, input_size=784, hidden_size=128, num_classes=10):
        super(DigitClassifier, self).__init__()

        # Input → Hidden
        self.fc1 = nn.Linear(input_size, hidden_size)

        # Hidden → Output
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Flatten input image (batch_size, 1, 28, 28) → (batch_size, 784)
        x = x.view(x.size(0), -1)

        # Hidden layer with ReLU activation
        x = self.fc1(x)
        x = F.relu(x)

        # Output layer (raw scores / logits)
        x = self.fc2(x)

        return x

model = DigitClassifier()

sample_input = torch.randn(32, 1, 28, 28)  # batch of 32 images
output = model(sample_input)

print(output.shape)  # Should be [32, 10]

torch.Size([32, 10])


**3. Train the model using Cross-Entropy Loss and an optimizer such as SGD or Adam for multiple epochs, printing the
training loss during training. After training, evaluate the model on the test dataset by computing and displaying the
classification accuracy.**


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

# Initialize model
model = DigitClassifier(input_size=784, hidden_size=128, num_classes=10)

# Loss function
criterion = nn.CrossEntropyLoss()

# Choose optimizer (Adam or SGD)
optimizer = optim.Adam(model.parameters(), lr=0.001)
# optimizer = optim.SGD(model.parameters(), lr=0.01)

num_epochs = 5

# Training Phase

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Training Loss: {epoch_loss:.4f}")

# Testing / Evaluation Phase

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\nTest Accuracy: {accuracy:.2f}%")

Epoch [1/5], Training Loss: 0.2570
Epoch [2/5], Training Loss: 0.1124
Epoch [3/5], Training Loss: 0.0775
Epoch [4/5], Training Loss: 0.0597
Epoch [5/5], Training Loss: 0.0459

Test Accuracy: 97.57%
